In [ ]:
#!/usr/bin/env python
# coding: utf-8

import os
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from tqdm import tqdm
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim
import numpy as np
import random
from pytorch_msssim import ssim as ssim_loss_fn
from skimage.exposure import match_histograms
import csv

# === Set Random Seeds ===
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# === Loss Function ===
def l1_ssim_loss(pred, target, alpha=0.75):
    pred = torch.nan_to_num(pred, nan=0.0, posinf=1.0, neginf=0.0)
    target = torch.nan_to_num(target, nan=0.0, posinf=1.0, neginf=0.0)
    pred = torch.clamp(pred, 0.0, 1.0)
    target = torch.clamp(target, 0.0, 1.0)

    l1 = F.l1_loss(pred, target)
    try:
        ssim_val = ssim_loss_fn(pred, target, data_range=1.0)
        if torch.numel(ssim_val) > 1:
            ssim_val = ssim_val.mean()
    except Exception:
        ssim_val = torch.tensor(0.0, device=pred.device)

    ssim_loss = 1 - ssim_val
    loss = alpha * l1 + (1 - alpha) * ssim_loss
    return loss

# === Optimizer & Scheduler ===
def configure_optimizer(model):
    return torch.optim.Adam(model.parameters(), lr=1e-4, betas=(0.5, 0.9), weight_decay=0)

def cosine_lr_with_warmup(optimizer, total_epochs, warmup_epochs=5):
    from torch.optim.lr_scheduler import SequentialLR, LinearLR, CosineAnnealingLR
    warmup = LinearLR(optimizer, start_factor=0.1, total_iters=warmup_epochs)
    cosine = CosineAnnealingLR(optimizer, T_max=max(1, total_epochs - warmup_epochs))
    return SequentialLR(optimizer, schedulers=[warmup, cosine], milestones=[warmup_epochs])

def count_trainable_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def save_checkpoint(model, optimizer, scheduler, epoch, save_path,
                    train_losses, val_losses, psnr_history, ssim_history):
    os.makedirs(save_path, exist_ok=True)
    torch.save({
        'epoch': epoch + 1,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'train_losses': train_losses,
        'val_losses': val_losses,
        'psnr': psnr_history,
        'ssim': ssim_history
    }, os.path.join(save_path, 'checkpoint.pt'))

def load_checkpoint(model, optimizer, scheduler, checkpoint_path, device):
    if os.path.isfile(checkpoint_path):
        checkpoint = torch.load(checkpoint_path, map_location=device)
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
        return (checkpoint['epoch'],
                checkpoint.get('train_losses', []),
                checkpoint.get('val_losses', []),
                checkpoint.get('psnr', []),
                checkpoint.get('ssim', []))
    return 0, [], [], [], []

# === Visualization helpers ===
def overlay_region_maps(ldct_img, region_maps):
    """
    ldct_img: (H,W) numpy
    region_maps: (R,H,W) numpy (intensity maps)
    returns fig saved externally
    """
    R = region_maps.shape[0]
    fig, axes = plt.subplots(1, R+1, figsize=(3*(R+1), 3))
    axes[0].imshow(ldct_img, cmap='gray')
    axes[0].set_title("LDCT")
    axes[0].axis('off')
    for r in range(R):
        # use color overlay for the region (normalize region map for display)
        rm = region_maps[r]
        if rm.max() > 0:
            disp = (rm - rm.min()) / (rm.max() - rm.min() + 1e-8)
        else:
            disp = rm
        axes[r+1].imshow(ldct_img, cmap='gray')
        axes[r+1].imshow(disp, cmap='magma', alpha=0.5)
        axes[r+1].set_title(f"Region {r+1}")
        axes[r+1].axis('off')
    plt.tight_layout()
    return fig

def save_gamma_beta_per_region(gamma, beta, region_maps, out_dir, epoch, prefix="gamma"):
    """
    gamma, beta: (B,C,H,W)
    region_maps: (B,R,H,W) with intensities (used as soft masks to aggregate)
    We'll compute region-weighted average of gamma per feature channel and save a small plot.
    """
    os.makedirs(out_dir, exist_ok=True)
    B, C, H, W = gamma.shape
    B2, R, H2, W2 = region_maps.shape
    if B != B2:
        # mismatch shapes; skip
        return

    g = gamma[0].cpu().numpy()  # (C,H,W)
    b = beta[0].cpu().numpy()
    rm = region_maps[0].cpu().numpy()  # (R,H,W)

    channel_means_by_region = np.zeros((R, C), dtype=np.float32)
    for r in range(R):
        mask = rm[r]
        denom = mask.sum() + 1e-8
        for c in range(C):
            channel_means_by_region[r, c] = (g[c] * mask).sum() / denom

    max_plot_channels = min(64, C)
    plt.figure(figsize=(12, 2 + R*0.25))
    plt.imshow(channel_means_by_region[:, :max_plot_channels], aspect='auto')
    plt.colorbar()
    plt.title(f"Epoch {epoch+1} - channel means (regions x channels)")
    plt.xlabel("feature channel (truncated)")
    plt.ylabel("region")
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, f"{prefix}_region_channelmeans_epoch_{epoch+1}.png"), dpi=150)
    plt.close()

# === Training Loop ===
def train_model_2d(model, train_loader, val_loader, num_epochs, device, save_path, resume=True):
    model = model.to(device)
    optimizer = configure_optimizer(model)
    scheduler = cosine_lr_with_warmup(optimizer, total_epochs=num_epochs)
    checkpoint_path = os.path.join(save_path, 'checkpoint.pt') if save_path else None
    visuals_dir = os.path.join(save_path, "visuals")
    film_vis_dir = os.path.join(visuals_dir, "film")
    masks_vis_dir = os.path.join(visuals_dir, "masks")
    history_path = os.path.join(save_path, "training_history.npz") if save_path else "training_history.npz"

    start_epoch = 0

    train_losses = []
    val_losses = []
    psnr_history = []
    ssim_history = []

    if resume and checkpoint_path and os.path.exists(checkpoint_path):
        checkpoint = torch.load(checkpoint_path, map_location=device)

        start_epoch, train_losses, val_losses, psnr_history, ssim_history = \
            load_checkpoint(model, optimizer, scheduler, checkpoint_path, device)
    if os.path.exists(history_path):
        history_data = np.load(history_path)
        train_losses = history_data['train_losses'].tolist()
        val_losses = history_data['val_losses'].tolist()
        psnr_history = history_data['psnr'].tolist()
        ssim_history = history_data['ssim'].tolist()
        print(f"Loaded training history up to epoch {start_epoch}")


    torch.backends.cudnn.benchmark = True

    for epoch in range(start_epoch, num_epochs):
        model.train()
        train_loss = 0.0
        train_count = 0
        print(f"\n[Epoch {epoch+1}/{num_epochs}]")

        for batch_idx, batch in enumerate(tqdm(train_loader, desc="Training")):
            if len(batch) == 4:
                ldct, noise_map, ndct, region_maps = batch
            else:
                ldct, noise_map, ndct = batch
                region_maps = None

            ldct = ldct.to(device, non_blocking=True)
            noise_map = noise_map.to(device, non_blocking=True)
            ndct = ndct.to(device, non_blocking=True)
            if region_maps is not None:
                region_maps = region_maps.to(device, non_blocking=True)

            optimizer.zero_grad()
            output = model(ldct, noise_map=noise_map, region_maps=region_maps)

            output = torch.nan_to_num(output, nan=0.0, posinf=1.0, neginf=0.0)
            output = torch.clamp(output, 0.0, 1.0)

            loss = l1_ssim_loss(output, ndct)
            if torch.isnan(loss) or torch.isinf(loss):
                print(f"[WARN] NaN/Inf loss epoch {epoch+1} batch {batch_idx} -> skipping")
                continue

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            train_loss += loss.item() * ldct.size(0)
            train_count += ldct.size(0)

            # Save visualizations for first batch of epoch
            if batch_idx == 0 and save_path is not None:
                os.makedirs(masks_vis_dir, exist_ok=True)
                ldct_np = ldct[0,0].cpu().numpy()
                region_maps_np = region_maps[0].cpu().numpy() if region_maps is not None else np.zeros((1, ldct_np.shape[0], ldct_np.shape[1]))
                fig = overlay_region_maps(ldct_np, region_maps_np)
                fig.savefig(os.path.join(masks_vis_dir, f"regions_epoch_{epoch+1}.png"), dpi=150)
                plt.close(fig)

                try:
                    local_film = model.decoder.res1.local_film
                    with torch.no_grad():
                        latent, skips = model.encoder(ldct, noise_map)
                        up = F.interpolate(latent, scale_factor=2, mode='bilinear', align_corners=False)
                        features_for_res1 = up  

                        gamma, beta = local_film.extract_region_conditioned_gamma_beta(features_for_res1, region_maps)
                        if gamma is not None:
                            region_maps_resized = F.interpolate(region_maps, size=gamma.shape[2:], mode='bilinear', align_corners=False)
                            save_gamma_beta_per_region(gamma, beta, region_maps_resized, film_vis_dir, epoch, prefix="gamma")
                except Exception as e:
                    print(f"[WARN] could not extract gamma/beta for visualization: {e}")



        scheduler.step()
        avg_train_loss = train_loss / train_count if train_count > 0 else float('nan')
        train_losses.append(avg_train_loss)

        # Validation
        model.eval()
        val_loss, psnr_vals, ssim_vals = 0.0, [], []
        val_count = 0
        with torch.no_grad():
            for batch in tqdm(val_loader, desc="Validation"):
                if len(batch) == 4:
                    ldct, noise_map, ndct, region_maps = batch
                else:
                    ldct, noise_map, ndct = batch
                    region_maps = None

                ldct = ldct.to(device, non_blocking=True)
                noise_map = noise_map.to(device, non_blocking=True)
                ndct = ndct.to(device, non_blocking=True)
                if region_maps is not None:
                    region_maps = region_maps.to(device, non_blocking=True)

                output = model(ldct, noise_map=noise_map, region_maps=region_maps)
                output = torch.nan_to_num(output, nan=0.0, posinf=1.0, neginf=0.0)
                output = torch.clamp(output, 0.0, 1.0)

                loss = l1_ssim_loss(output, ndct)
                if torch.isnan(loss) or torch.isinf(loss):
                    continue

                val_loss += loss.item() * ldct.size(0)
                val_count += ldct.size(0)

                for i in range(ldct.size(0)):
                    out_np = output[i, 0].cpu().numpy()
                    gt_np = ndct[i, 0].cpu().numpy()
                    if not (np.isfinite(out_np).all() and np.isfinite(gt_np).all()):
                        continue
                    try:
                        out_np = match_histograms(out_np, gt_np, channel_axis=None)
                        out_np = np.clip(out_np, 0.0, 1.0)
                        gt_np = np.clip(gt_np, 0.0, 1.0)

                        psnr_vals.append(psnr(gt_np, out_np, data_range=1.0))
                        ssim_vals.append(ssim(gt_np, out_np, data_range=1.0))
                    except Exception:
                        continue

        avg_val_loss = val_loss / val_count if val_count > 0 else float('nan')
        avg_psnr = np.mean(psnr_vals) if psnr_vals else 0.0
        avg_ssim = np.mean(ssim_vals) if ssim_vals else 0.0

        val_losses.append(avg_val_loss)
        psnr_history.append(avg_psnr)
        ssim_history.append(avg_ssim)

        print(f"Avg Val Loss: {avg_val_loss:.6f} | PSNR: {avg_psnr:.2f} dB | SSIM: {avg_ssim:.4f}")

        if save_path:
            save_checkpoint(model, optimizer, scheduler, epoch, save_path,
                train_losses, val_losses, psnr_history, ssim_history)
            if (epoch + 1) % 10 == 0:
                metrics_file = os.path.join(save_path, "metrics_every_10_epochs.csv")
                file_exists = os.path.isfile(metrics_file)
                with open(metrics_file, mode='a', newline='') as csvfile:
                    writer = csv.writer(csvfile)
                    if not file_exists:
                        writer.writerow(["Epoch", "Avg_PSNR", "Avg_SSIM"])
                    writer.writerow([epoch + 1, round(avg_psnr, 4), round(avg_ssim, 4)])

        np.savez(history_path, train_losses=train_losses, val_losses=val_losses,
                 psnr=psnr_history, ssim=ssim_history)
        print(f"\nSaved training history to {history_path}")

    # final loss plot
    plt.figure(figsize=(10,5))
    plt.plot(train_losses, label='Train Loss')
    plt.plot(val_losses, label='Val Loss')
    plt.legend()
    plt.grid()
    if save_path:
        plt.savefig(os.path.join(save_path, "loss_curve.png"))
    plt.close()


if __name__ == '__main__':
    from dataset import CT2DSliceDataset
    from model import DenoisingUNet2D_HybridFiLM

    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    train_dataset = CT2DSliceDataset("training 1mm")
    val_dataset = CT2DSliceDataset("validation 1mm")

    train_loader = DataLoader(train_dataset, batch_size=7, shuffle=True, num_workers=8, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=7, shuffle=False, num_workers=8, pin_memory=True)

    model = DenoisingUNet2D_HybridFiLM(num_regions=3).to(device)
    print(f"Total trainable params: {count_trainable_parameters(model):,}")

    save_path = "output_dir"
    os.makedirs(save_path, exist_ok=True)
    train_model_2d(model=model, train_loader=train_loader, val_loader=val_loader,
                   num_epochs=100, device=device, save_path=save_path, resume=True)
